# Data Preprocessing Notebook
This notebook combines and organizes all preprocessing logic from `preprocess.py`, `cleaning.py`, and `Cleaning_and_Preprocessing.ipynb`.
The goal is to clean, transform, and prepare the data for analysis and modeling.

## Introduction and Imports
This section imports all necessary libraries and provides an overview of the preprocessing steps.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
import requests
import re
import json

## Data Loading
Load the dataset for preprocessing. This includes reading data from JSON and CSV files.

In [ ]:
# Load data
def load_data(json_file):
    # Load JSON data
    df_json = pd.read_json(json_file)
    df_json.to_csv(csv, index=False)
    return csv

# Example usage
unclean_csv = load_data("../Data/property_data.json")

## Data Cleaning
This section handles missing values, removes duplicates, and cleans columns. It also includes logic for standardizing and transforming data.

In [ ]:
# Data Cleaning Functions
def clean_data(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    # Display basic information before cleaning
    print("\nBasic Information BEFORE Cleaning:")
    print(df.info())

    # Drop rows with missing prices
    df = df.dropna(subset=['price'])

    # Clean and standardize the 'price' column
    df['price'] = df['price'].replace(r'[^0-9]', '', regex=True)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')  # Convert to numeric, invalid values become NaN

    # Convert price to numeric
    df['price'] = (
        df['price']
        .str.replace('LYD', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
        .astype(str)
        .str.extract('(\\d+)').astype(float)
    )
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

    # Filter data based on realistic values
    df = df[(df['price'] >= 15000) & (df['price'] <= 15000000)]
    df = df[(df['surface_area'] >= 50) & (df['surface_area'] <= 1000)]

    # Clean and convert surface area to numeric
    df['surface_area'] = (
        df['surface_area']
        .astype(str)
        .str.replace(' meter square', '', regex=False)
        .str.replace(' sqm', '', regex=False)
        .apply(pd.to_numeric, errors='coerce')
    )

    # Create price per meter square column
    df['price_per_meter_square'] = df['price'] / df['surface_area']


    # Convert 'attributes' column to strings temporarily to handle duplicates
    df['attributes'] = df['attributes'].apply(lambda x: json.dumps(x) if isinstance(x, dict) else x)

    # Remove duplicate rows
    df = df.drop_duplicates()


    # Parse JSON-like strings in the 'attributes' column into separate columns
    def parse_attributes(attr):
        try:
            return json.loads(attr) if isinstance(attr, str) else {}
        except json.JSONDecodeError:
            return {}

    attributes_parsed = df['attributes'].apply(parse_attributes)
    attributes_df = pd.json_normalize(attributes_parsed)
    df = pd.concat([df, attributes_df], axis=1)
    df = df.drop(columns=['attributes'])  # Drop the original 'attributes' column

    # Validate and clean the 'location' column
    df['location'] = df['location'].apply(lambda x: x if isinstance(x, str) and x.startswith('http') else None)

    
    # Extract latitude and longitude from location links
    df = extract_lat_long_from_location(df, 'location')
    # Drop unnecessary columns
    df.drop(columns=['location'], errors='ignore', inplace=True)

    # Remove impossible location values (Libya bounds)
    df = df[(df['latitude'] >= 19.5) & (df['latitude'] <= 33)]
    df = df[(df['longitude'] >= 9) & (df['longitude'] <= 25)]
    
    # Standardize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # Drop unnecessary columns
    columns_to_drop = [
        'category', 'reference_id', 'payment_method', 'main_amenities', 'nearby', 'building_age',
        'additional_amenities', 'land_area', 'property_status', 'country',
        'real_estate_type', 'floor', 'zoned_for', 'number_of_floors', 'url', 'description'
    ]
    df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

    # Remove columns with zero non-null values
    df = df.loc[:, df.notna().any()]

    #remove all catagories that aren't properties for sale
    df = df[df['category'] == 'Property For Sale']

    # Remove specific subcategories
    df = df[df['subcategory'] != 'Farms & Chalets for Sale']
    df = df[df['subcategory'] != 'Whole Building for Sale']
    df['subcategory'] = df['subcategory'].replace({
        'Apartments for Sale': 'Apartment',
        'Villas for Sale': 'Villa',
        'Homes for Sale': 'House'
    })

    # Convert to boolean
    boolean_columns = {'furnished?': 'Furnished', 'property_mortgaged?': 'Yes'}
    for col, true_value in boolean_columns.items():
        df[col] = df[col].apply(lambda x: x == true_value)

    # Handle missing values
    fill_value_columns = {'facade': 'Unknown', 'property_mortgaged?': 'No'}
    df.fillna(value=fill_value_columns, inplace=True)

    # Standardize categorical columns
    bedroom_mapping = {
        '7 +': 7, 'More Than 6 Bedrooms': 7, 'Studio': 1, '1 Bedroom': 1, '2 Bedrooms': 2,
        '3 Bedrooms': 3, '4 Bedrooms': 4, '5 Bedrooms': 5
    }
    bathroom_mapping = {
        '7 +': 7, 'More Than 6 Bathrooms': 7, 'One Bathroom': 1, '2 Bathrooms': 2,
        '3 Bathrooms': 3, '4 Bathrooms': 4, '5 Bathrooms': 5
    }
    df['bedrooms'] = df['bedrooms'].replace(bedroom_mapping).astype(float)
    df['bathrooms'] = df['bathrooms'].replace(bathroom_mapping).astype(float)

    
    # Display basic information after cleaning
    print("\nBasic Information After Cleaning:")
    print(df.info())

    df.to_csv(output_csv, index=False)

# Example usage
clean_data("../Data/cleaned_data.csv", "../Data/processed_data.csv")

extract_lat_long_from_location was used to extract the Latitude and Longitude from the location google map links

In [ ]:
def extract_lat_long_from_location(df, location_column):
    latitudes = []
    longitudes = []

    for link in df[location_column]:
        try:
            # Parse the URL and extract the query parameters
            parsed_url = urlparse(link)
            query_params = parse_qs(parsed_url.query)

            # Extract the 'query' parameter containing the coordinates
            if 'query' in query_params:
                coords = query_params['query'][0].split(',')
                if len(coords) == 2:
                    latitudes.append(float(coords[0]))
                    longitudes.append(float(coords[1]))
                else:
                    latitudes.append(None)
                    longitudes.append(None)
            else:
                latitudes.append(None)
                longitudes.append(None)
        except Exception as e:
            latitudes.append(None)
            longitudes.append(None)

    df['latitude'] = latitudes
    df['longitude'] = longitudes
    return df

Since the Scraped data was entered by humans, it is prone to human error. The following code attempted to normalise the names of cities

In [ ]:
def standardize_city_name(name):
    if not isinstance(name, str) or name.lower() == 'nan':
        return "Other"
    
    #Basic Normalization
    name = name.lower().strip()
    name = name.replace('-', ' ').replace("'", "")

    # 2. PHONETIC NORMALIZATION (The "Sounds-Like" Step)
    # This step handles the e/i/ee and h/ah differences before the mapping
    def normalize_sounds(text):
        # Convert ee, i, y to a single 'i'
        text = re.sub(r'ee|y|e', 'i', text)
        # Handle terminal 'h' or 'ah' (Common in Arabic names like Humaidah)
        text = re.sub(r'(ah|h)$', 'a', text)
        # Standardize Al/An/As/Ar prefixes
        text = re.sub(r'^(an|as|ar|at|ad|az|ash)\s+', 'al ', text)
        # Remove double consonants (e.g., Hammoudat -> Hamodat)
        text = re.sub(r'(.)\1+', r'\1', text)
        return text.strip()

    # 3. Comprehensive Mapping based on your latest list
    # The keys here are the "Standard" names you want for your project
    mapping = {
        'tripoli': ['ain zara', 'janzour', 'Qasr Bin Ghashir', 'soug al juma’aa', 'tajoura', 'souq al jomoa', 'souq al jumaa', 'tripoli', 'طرابلس'],
        'al khoms': ['al khoms', 'al khums'],
        'misratah': ['misrata', 'misratah', 'misurata'],
        'al zawiya': ['الزاويه جودايم'],
        'other': ['l.l']
    }

    # First, try an exact match with the normalized version
    normalized_name = normalize_sounds(name)
    
    for standard, variations in mapping.items():
        # Normalize all variations for comparison
        norm_variations = [normalize_sounds(v) for v in variations]
        
        # Check if the input (normalized) matches any variation (normalized)
        if normalized_name in norm_variations:
            return standard
            
        # Partial match (if 'ainzara' is found within 'ainzara road')
        for nv in norm_variations:
            if nv in normalized_name or normalized_name in nv:
                return standard

    return name.strip().title()

For detecting and removing outliers

In [ ]:
def outlier_detection(input_csv, category_column, value_column, save_extention):
    """Graphs a box and whiskers plot for value_column grouped by category_column."""
    df = pd.read_csv(input_csv)
    plt.figure(figsize=(12, 8))
    sns.boxplot(x=category_column, y=value_column, data=df)
    plt.title('Price Distribution by: ' + category_column)
    plt.xlabel(category_column)
    plt.ylabel(value_column)
    plt.xticks(rotation=45)
    plt.savefig(f"graphs/box_plot_{value_column}_by_{category_column + save_extention}.png")
    plt.close()

def remove_outliers_by_category(input_csv, output_csv, category_column, value_column):
    """Removes outliers in the value_column within each category of the category_column."""
    df = pd.read_csv(input_csv)

    def remove_outliers(group):
        q1 = group[value_column].quantile(0.25)
        q3 = group[value_column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        return group[(group[value_column] >= lower_bound) & (group[value_column] <= upper_bound)]

    filtered_df = df.groupby(category_column, group_keys=False).apply(remove_outliers)
    filtered_df.to_csv(output_csv, index=False)
    print(f"Outliers removed and data saved to {output_csv}")

## Feature Engineering
This section creates new features and transforms existing ones, such as log transformations and derived columns.

In [ ]:
# Feature Engineering Functions
def log_transform(df):
    df['price'] = np.log1p(df['price'])
    df['surface_area'] = np.log1p(df['surface_area'])
    df['price_per_meter_square'] = np.log1p(df['price_per_meter_square'])
    return df

# Example usage
df = log_transform(df)

## Encoding and Transformations
This section encodes categorical variables and normalizes data for analysis.

In [ ]:
# Encoding Functions
def encode_categorical(df):
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    categorical_cols = df.select_dtypes(include=['object']).columns
    encoded_data = encoder.fit_transform(df[categorical_cols])
    encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols))

    # Combine with numerical data
    df = pd.concat([df.select_dtypes(exclude=['object']), encoded_df], axis=1)
    return df

# Example usage
df = encode_categorical(df)

## Final Dataset Export
This section saves the cleaned and processed dataset to a CSV file.

In [ ]:
# Save Processed Data
def save_data(df, output_file):
    df.to_csv(output_file, index=False)
    print(f"Data saved to {output_file}")

# Example usage
save_data(df, "../Data/processed_data.csv")